# Drone Object Detection Using YOLO26n

## Project overview
This project uses the YOLO26n object-detection model fine-tuned on the VisDrone2019-DET dataset. The model detects 10 object categories in drone-view images.

**Dataset:** VisDrone2019-DET  
**Model:** YOLO26n  
**Selected evaluation image size:** 1536  
**Prediction confidence threshold:** 0.25

The trained model checkpoint is loaded from Google Drive. Training experiments and validation were performed in Google Colab.

## 1. Install dependencies and connect Google Drive

Run this notebook in Google Colab. Make sure the dataset and trained model are available in your Google Drive / Colab environment.

In [ ]:
%pip install -q ultralytics
from google.colab import drive
drive.mount('/content/drive')

## 2. Load the trained model

The final selected checkpoint is stored in the `Best_Model` folder. Update the path if your Drive folder is different.

In [ ]:
from ultralytics import YOLO

MODEL_PATH = '/content/drive/MyDrive/YOLO_Drone_Project/Best_Model/best_model.pt'
model = YOLO(MODEL_PATH)

print("Model loaded successfully.")
print("Model:", model.model_name if hasattr(model, "model_name") else "YOLO26n")

## 3. Validate the model

Validation uses the VisDrone validation split and a maximum of 1000 detections per image. The dataset YAML must be available at the path supplied below.

In [ ]:
DATA_YAML = '/content/datasets/VisDrone/VisDrone.yaml'  # Update if needed

metrics = model.val(
    data=DATA_YAML,
    imgsz=1536,
    max_det=1000,
    plots=True
)

print(f"Precision: {metrics.box.mp * 100:.2f}%")
print(f"Recall: {metrics.box.mr * 100:.2f}%")
print(f"mAP50: {metrics.box.map50 * 100:.2f}%")
print(f"mAP50-95: {metrics.box.map * 100:.2f}%")

### Selected validation results

The following values are from the saved notebook's validation run of the selected model at image size 1536:

| Metric | Result |
|---|---:|
| Precision | 52.89% |
| Recall | 45.76% |
| mAP50 | 45.84% |
| mAP50-95 | 27.06% |

These are validation-set metrics, not accuracy measured on the separate test-challenge images.

## 4. Detect objects in a sample image

Change `IMAGE_PATH` to the image you want to test. The prediction displays bounding boxes, class names, and confidence scores.

In [ ]:
from IPython.display import display
from PIL import Image

IMAGE_PATH = '/content/datasets/VisDrone/images/val/0000271_02401_d_0000385.jpg'

results = model.predict(
    source=IMAGE_PATH,
    imgsz=1536,
    conf=0.25,
    save=True,
    show_labels=True,
    show_conf=True,
    line_width=2
)

display(Image.fromarray(results[0].plot()[..., ::-1]))

## 5. Test on unseen drone images

The following cell runs inference on a folder of test images. In the original project, 10 randomly selected images from the VisDrone2019-DET test-challenge set were tested. The test-challenge images do not provide public ground-truth labels for this project workflow, so these results are used for visual demonstration rather than calculating precision or mAP.

In [ ]:
from pathlib import Path

TEST_IMAGE_FOLDER = '/content/VisDrone_test/VisDrone2019-DET-test-challenge/images'
OUTPUT_FOLDER = '/content/drive/MyDrive/YOLO_Drone_Project/final_test_results'

test_images = sorted(Path(TEST_IMAGE_FOLDER).glob('*.jpg'))
print("Images available:", len(test_images))

# Select up to 10 images for demonstration
sample_images = test_images[:10]

test_results = model.predict(
    source=[str(path) for path in sample_images],
    imgsz=1536,
    conf=0.25,
    save=True,
    show_labels=True,
    show_conf=True,
    line_width=2,
    project='/content/drive/MyDrive/YOLO_Drone_Project',
    name='final_test_results',
    exist_ok=True
)

print("Images tested:", len(sample_images))

## 6. Display saved prediction images

In [ ]:
from IPython.display import display, Image
import glob

prediction_files = glob.glob(
    '/content/drive/MyDrive/YOLO_Drone_Project/final_test_results/*.jpg'
)

print("Prediction images found:", len(prediction_files))
for file in prediction_files:
    display(Image(filename=file, width=700))

## 7. Project notes and limitations

- The model was evaluated on the VisDrone validation split.
- The selected inference image size is 1536 and the selected confidence threshold is 0.25.
- A lower confidence threshold can retain more detections, but may also produce more false positives.
- The 10 unseen test-challenge images are used for qualitative demonstration only because corresponding ground-truth annotations were not available in this workflow.
- Detection performance can vary with image quality, object size, crowding, and hardware.

## Conclusion

The project demonstrates drone-view object detection using a YOLO26n model fine-tuned on VisDrone2019-DET. The final model was evaluated at multiple image sizes, and 1536 was selected based on the validation results. The selected model was then used to generate predictions on unseen drone images.